# 20 — Consolidated Target Review Pack V1

This notebook consolidates the calibrated V2 target model into practical review files.

It does **not** change the scoring model. It takes the V2 watchlist outputs and creates:

- a consolidated ward review list with flags for each strategic lane;
- clean, caveated, breakthrough and demographic-build CSVs;
- council-level briefing summaries;
- map-ready outputs for the four opportunity lanes;
- manual political-intelligence fields for SDP/member/candidate review.

The purpose is to move from scoring outputs to an operational review pack.

## 20.1 Setup

The notebook assumes the standard project structure:

```text
Electoral_Tribes/
  data/
    processed/
      target_model_v2/
      target_review_pack_v1/
```

If you ran the V2 scoring notebook with a different output folder, update `TARGET_MODEL_DIR` below.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import shutil
from datetime import date

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name.lower() == "notebooks" else NOTEBOOK_DIR

DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
TARGET_MODEL_DIR = PROCESSED_DIR / "target_model_v2"
OUTPUT_DIR = PROCESSED_DIR / "target_review_pack_v1"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Scope controls
DEFAULT_SCOPE = "north_west"
GENERATE_ALL_AVAILABLE_OUTPUTS = True

print("Project directory:", PROJECT_DIR)
print("Target model directory:", TARGET_MODEL_DIR)
print("Output directory:", OUTPUT_DIR)
print("Run date:", date.today())

Project directory: c:\Users\keena\Documents\Electoral_Tribes
Target model directory: c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_model_v2
Output directory: c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_review_pack_v1
Run date: 2026-05-27


## 20.2 Helper functions

These functions keep the notebook robust when a file has a slightly different name or a column is unavailable.

In [2]:
def find_file(filename, search_dirs=None, required=True):
    """Find a CSV by filename in likely directories."""
    if search_dirs is None:
        search_dirs = [TARGET_MODEL_DIR, PROCESSED_DIR, PROJECT_DIR, Path.cwd(), Path("/mnt/data")]
    
    for folder in search_dirs:
        path = folder / filename
        if path.exists():
            return path
    
    # Recursive fallback inside processed folder
    for folder in [TARGET_MODEL_DIR, PROCESSED_DIR]:
        if folder.exists():
            matches = list(folder.rglob(filename))
            if matches:
                return matches[0]
    
    if required:
        raise FileNotFoundError(f"Could not find required file: {filename}")
    return None


def read_csv_file(filename, required=True):
    path = find_file(filename, required=required)
    if path is None:
        print(f"Optional file missing: {filename}")
        return None
    print("Loaded:", path)
    return pd.read_csv(path, low_memory=False)


def norm_bool(series):
    return series.fillna(False).astype(str).str.lower().isin(["true", "1", "yes", "y"])


def key_series(df):
    if "WD25CD" not in df.columns:
        raise KeyError("WD25CD missing from dataframe")
    return df["WD25CD"].astype(str).str.strip()


def safe_col(df, col, default=np.nan):
    if col not in df.columns:
        df[col] = default
    return df[col]


def first_existing(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


def pct_fmt(x):
    if pd.isna(x):
        return ""
    return f"{x:.1%}"


def score_fmt(x):
    if pd.isna(x):
        return ""
    return f"{x:.1f}"

## 20.3 Load V2 watchlist and score files

The base file should ideally be `target_score_components_ward25_all_available_v2.csv` or `initial_watchlist_scores_ward25_all_available_v2.csv`.

If neither exists, the notebook can still build a partial pack from the watchlist files, but the full consolidated review list needs the all-ward score file.

In [3]:
# Preferred all-ward V2 score file.
base_candidates = [
    "target_score_components_ward25_all_available_v2.csv",
    "initial_watchlist_scores_ward25_all_available_v2.csv",
    "top_100_initial_watchlist_all_available_v2.csv",
]

base = None
base_path = None
for filename in base_candidates:
    path = find_file(filename, required=False)
    if path is not None:
        base = pd.read_csv(path, low_memory=False)
        base_path = path
        break

if base is None:
    raise FileNotFoundError("No usable V2 base score file found.")

print("Base score file:", base_path)
print("Base shape:", base.shape)

clean = read_csv_file("clean_watchlist_review_ab_north_west_v2.csv", required=False)
caveated = read_csv_file("caveated_watchlist_review_ab_north_west_v2.csv", required=False)
demographic_build = read_csv_file("demographic_build_watchlist_north_west_v2.csv", required=False)
breakthrough = read_csv_file("breakthrough_complacency_watchlist_north_west_v2.csv", required=False)
top100_nw = read_csv_file("top_100_initial_watchlist_north_west_v2.csv", required=False)
council_v2 = read_csv_file("council_review_summary_north_west_v2.csv", required=False)

# Optional all-region files, if generated by Notebook 18b.
all_clean = read_csv_file("clean_watchlist_review_ab_all_available_v2.csv", required=False)
all_caveated = read_csv_file("caveated_watchlist_review_ab_all_available_v2.csv", required=False)
all_demo = read_csv_file("demographic_build_watchlist_all_available_v2.csv", required=False)
all_breakthrough = read_csv_file("breakthrough_complacency_watchlist_all_available_v2.csv", required=False)
all_top100 = read_csv_file("top_100_initial_watchlist_all_available_v2.csv", required=False)

Base score file: c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_model_v2\outputs\target_score_components_ward25_all_available_v2.csv
Base shape: (7572, 36)
Loaded: c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_model_v2\watchlists\clean_watchlist_review_ab_north_west_v2.csv
Loaded: c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_model_v2\watchlists\caveated_watchlist_review_ab_north_west_v2.csv
Loaded: c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_model_v2\watchlists\demographic_build_watchlist_north_west_v2.csv
Loaded: c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_model_v2\watchlists\breakthrough_complacency_watchlist_north_west_v2.csv
Loaded: c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_model_v2\watchlists\top_100_initial_watchlist_north_west_v2.csv
Loaded: c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_model_v2\review\council_review_summary_north_west_v2.csv
Load

## 20.4 Prepare scoped base tables

The North West is the default operational scope. National/all-available outputs can be generated if the required all-region files exist.

In [4]:
base = base.copy()
base["WD25CD"] = key_series(base)

if "analysis_region" not in base.columns:
    base["analysis_region"] = np.where(base.get("scope_north_west", False), "North West", np.nan)

if "country_inferred" in base.columns:
    base["analysis_region"] = np.where(
        base["analysis_region"].isna() & base["country_inferred"].eq("Wales"),
        "Wales",
        base["analysis_region"]
    )

if "scope_north_west" in base.columns:
    north_west_base = base[base["scope_north_west"].fillna(False).astype(bool)].copy()
else:
    north_west_base = base[base["analysis_region"].eq("North West")].copy()

print("North West base rows:", len(north_west_base))
print("All base rows:", len(base))

North West base rows: 825
All base rows: 7572


## 20.5 Add watchlist flags and strategic lanes

A ward can appear in more than one lane. For operational use, the notebook assigns a primary `strategic_lane` but also preserves all Boolean flags.

In [6]:
def keys_from(df):
    if df is None or len(df) == 0:
        return set()
    return set(key_series(df))

clean_keys = keys_from(clean)
caveated_keys = keys_from(caveated)
demo_keys = keys_from(demographic_build)
break_keys = keys_from(breakthrough)
top100_keys = keys_from(top100_nw)


def add_review_flags(df, clean_keys, caveated_keys, demo_keys, break_keys, top100_keys):
    df = df.copy()
    df["WD25CD"] = key_series(df)
    
    df["is_clean_watchlist"] = df["WD25CD"].isin(clean_keys)
    df["is_caveated_watchlist"] = df["WD25CD"].isin(caveated_keys)
    df["is_demographic_build"] = df["WD25CD"].isin(demo_keys)
    df["is_breakthrough_complacency"] = df["WD25CD"].isin(break_keys)
    df["is_top100_watchlist"] = df["WD25CD"].isin(top100_keys)
    
    def lane(row):
        if row["is_clean_watchlist"]:
            return "Clean Opportunity"
        if row["is_caveated_watchlist"]:
            return "Caveated Opportunity"
        if row["is_breakthrough_complacency"]:
            return "Breakthrough Build"
        if row["is_demographic_build"]:
            return "Long-Term Demographic Build"
        if str(row.get("review_band", "")).strip() in ["Review A", "Review B"]:
            return "General Watchlist"
        return "Monitor"
    
    def lane_priority(row):
        order = {
            "Clean Opportunity": 1,
            "Caveated Opportunity": 2,
            "Breakthrough Build": 3,
            "Long-Term Demographic Build": 4,
            "General Watchlist": 5,
            "Monitor": 6,
        }
        return order.get(row["strategic_lane"], 99)
    
    def secondary_lanes(row):
        lanes = []
        if row["is_clean_watchlist"]: lanes.append("Clean Opportunity")
        if row["is_caveated_watchlist"]: lanes.append("Caveated Opportunity")
        if row["is_breakthrough_complacency"]: lanes.append("Breakthrough Build")
        if row["is_demographic_build"]: lanes.append("Long-Term Demographic Build")
        return "; ".join(lanes)
    
    df["strategic_lane"] = df.apply(lane, axis=1)
    df["strategic_lane_priority"] = df.apply(lane_priority, axis=1)
    df["strategic_lane_flags"] = df.apply(secondary_lanes, axis=1)
    
    return df

review = add_review_flags(
    north_west_base,
    clean_keys=clean_keys,
    caveated_keys=caveated_keys,
    demo_keys=demo_keys,
    break_keys=break_keys,
    top100_keys=top100_keys,
)

review["strategic_lane"].value_counts(dropna=False)

strategic_lane
Monitor                        592
Breakthrough Build             120
Clean Opportunity               49
Long-Term Demographic Build     40
Caveated Opportunity            24
Name: count, dtype: int64

## 20.6 Add manual political intelligence fields

These fields are intentionally blank. They are for local knowledge, member/candidate intelligence, and later operational review.

In [7]:
manual_fields = {
    "candidate_known": False,
    "candidate_name": "",
    "candidate_strength_rating": "",
    "local_contact_known": False,
    "member_presence": "",
    "recent_sdp_activity": "",
    "local_issue_hook": "",
    "activist_accessibility": "",
    "delivery_practicality": "",
    "campaign_cost_estimate": "",
    "manual_priority": "",
    "manual_notes": "",
    "reviewed_by": "",
    "reviewed_date": "",
}

for col, default in manual_fields.items():
    if col not in review.columns:
        review[col] = default

# Useful text field for review meetings.
def model_summary(row):
    parts = []
    parts.append(f"Lane: {row.get('strategic_lane', '')}")
    if pd.notna(row.get("dominant_cluster_name", np.nan)):
        parts.append(f"Dominant tribe: {row.get('dominant_cluster_name')}")
    if pd.notna(row.get("latest_election_top_party_bucket", np.nan)):
        parts.append(f"Latest top party: {row.get('latest_election_top_party_bucket')}")
    if bool(row.get("has_major_caveat", False)):
        parts.append("Caveat present")
    return " | ".join(parts)

review["model_review_summary"] = review.apply(model_summary, axis=1)

## 20.7 Export consolidated and lane-specific review files

These are the main operational CSVs for human review.

In [8]:
preferred_cols = [
    "LAD25CD", "LAD25NM", "WD25CD", "WD25NM", "analysis_region",
    "strategic_lane", "strategic_lane_priority", "strategic_lane_flags",
    "is_clean_watchlist", "is_caveated_watchlist", "is_breakthrough_complacency", "is_demographic_build", "is_top100_watchlist",
    "initial_watchlist_score", "initial_watchlist_percentile", "review_band", "review_band_clean",
    "demographic_relevance_score", "electoral_opportunity_score", "political_openness_score", "breakthrough_complacency_score", "data_confidence_score",
    "dominant_cluster_name", "second_cluster_name", "dominant_cluster_share", "cluster_fragmentation_index",
    "latest_election_source_year", "latest_election_top_party_bucket", "latest_election_runner_up_party_bucket",
    "latest_election_margin_pct_allocated", "latest_election_party_fragmentation_index", "latest_election_effective_number_of_parties",
    "latest_election_con_share", "latest_election_lab_share", "latest_election_ld_share", "latest_election_green_share", "latest_election_reform_ukip_brexit_share", "latest_election_independent_share", "latest_election_sdp_share", "latest_election_other_share",
    "has_major_caveat", "boundary_caveat", "county_election_caveat", "target_model_ready", "data_confidence_note",
    "model_review_summary",
] + list(manual_fields.keys())

export_cols = [c for c in preferred_cols if c in review.columns]

review_export = review.sort_values(
    ["strategic_lane_priority", "initial_watchlist_score"],
    ascending=[True, False]
)[export_cols].copy()

review_export.to_csv(OUTPUT_DIR / "north_west_consolidated_target_review_v1.csv", index=False)

lane_files = {
    "Clean Opportunity": "north_west_clean_opportunity_wards_v1.csv",
    "Caveated Opportunity": "north_west_caveated_opportunity_wards_v1.csv",
    "Breakthrough Build": "north_west_breakthrough_build_wards_v1.csv",
    "Long-Term Demographic Build": "north_west_long_term_demographic_build_wards_v1.csv",
}

for lane, filename in lane_files.items():
    subset = review_export[review_export["strategic_lane"].eq(lane)].copy()
    subset.to_csv(OUTPUT_DIR / filename, index=False)
    print(f"{lane}: {len(subset)} rows -> {filename}")

print("Consolidated review rows:", len(review_export))
print("Saved:", OUTPUT_DIR / "north_west_consolidated_target_review_v1.csv")

Clean Opportunity: 49 rows -> north_west_clean_opportunity_wards_v1.csv
Caveated Opportunity: 24 rows -> north_west_caveated_opportunity_wards_v1.csv
Breakthrough Build: 120 rows -> north_west_breakthrough_build_wards_v1.csv
Long-Term Demographic Build: 40 rows -> north_west_long_term_demographic_build_wards_v1.csv
Consolidated review rows: 825
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_review_pack_v1\north_west_consolidated_target_review_v1.csv


## 20.8 Council briefing summaries

Council summaries support discussion at the level where campaign decisions are often made: where should candidate recruitment, local intelligence gathering, and initial campaign testing be prioritised?

In [9]:
def top_wards_for_lane(group, lane, n=5):
    sub = group[group["strategic_lane"].eq(lane)].copy()
    if sub.empty:
        return ""
    sub = sub.sort_values("initial_watchlist_score", ascending=False).head(n)
    return "; ".join(
        f"{r.WD25NM} ({r.initial_watchlist_score:.1f})"
        for r in sub.itertuples()
    )


def council_category(row):
    if row["clean_opportunity_count"] >= 3:
        return "Priority exploration council"
    if row["clean_opportunity_count"] >= 1 and row["breakthrough_build_count"] >= 3:
        return "Mixed opportunity and breakthrough council"
    if row["caveated_opportunity_count"] >= 3:
        return "Caveated opportunity council - verify ward-level evidence"
    if row["breakthrough_build_count"] >= 5:
        return "Breakthrough / long-build council"
    if row["demographic_build_count"] >= 5:
        return "Long-term demographic build council"
    if row["max_score"] >= 65:
        return "Selective ward-level opportunity"
    return "Monitor"

council_summary = (
    review
    .groupby(["LAD25CD", "LAD25NM", "analysis_region"], as_index=False)
    .agg(
        ward_count=("WD25CD", "nunique"),
        population=("population", "sum") if "population" in review.columns else ("WD25CD", "size"),
        mean_score=("initial_watchlist_score", "mean"),
        median_score=("initial_watchlist_score", "median"),
        max_score=("initial_watchlist_score", "max"),
        clean_opportunity_count=("is_clean_watchlist", "sum"),
        caveated_opportunity_count=("is_caveated_watchlist", "sum"),
        breakthrough_build_count=("is_breakthrough_complacency", "sum"),
        demographic_build_count=("is_demographic_build", "sum"),
        top100_count=("is_top100_watchlist", "sum"),
        mean_demographic_relevance=("demographic_relevance_score", "mean"),
        mean_electoral_opportunity=("electoral_opportunity_score", "mean"),
        mean_political_openness=("political_openness_score", "mean"),
        mean_breakthrough=("breakthrough_complacency_score", "mean"),
        major_caveat_count=("has_major_caveat", "sum") if "has_major_caveat" in review.columns else ("WD25CD", "size"),
    )
)

# Add top ward lists by lane.
for lane, colname in [
    ("Clean Opportunity", "top_clean_opportunity_wards"),
    ("Caveated Opportunity", "top_caveated_opportunity_wards"),
    ("Breakthrough Build", "top_breakthrough_build_wards"),
    ("Long-Term Demographic Build", "top_demographic_build_wards"),
]:
    lane_text = (
        review.groupby(["LAD25CD", "LAD25NM"], group_keys=False)
        .apply(lambda g: top_wards_for_lane(g, lane, n=5))
        .reset_index(name=colname)
    )
    council_summary = council_summary.merge(lane_text, on=["LAD25CD", "LAD25NM"], how="left")

council_summary["strategic_interpretation"] = council_summary.apply(council_category, axis=1)

council_summary = council_summary.sort_values(
    ["clean_opportunity_count", "caveated_opportunity_count", "breakthrough_build_count", "mean_score"],
    ascending=[False, False, False, False]
)

council_summary.to_csv(OUTPUT_DIR / "north_west_council_briefing_summary_v1.csv", index=False)

display(council_summary.head(20))

,LAD25CD,LAD25NM,analysis_region,ward_count,population,mean_score,median_score,max_score,clean_opportunity_count,caveated_opportunity_count,breakthrough_build_count,demographic_build_count,top100_count,mean_demographic_relevance,mean_electoral_opportunity,mean_political_openness,mean_breakthrough,major_caveat_count,top_clean_opportunity_wards,top_caveated_opportunity_wards,top_breakthrough_build_wards,top_demographic_build_wards,strategic_interpretation
31,E08000012,Liverpool,North West,64,64,50.576947,51.848641,72.235706,6,0,21,20,6,52.813460,42.354622,39.543399,37.463100,0,Old Swan East (72.2); Wavertree Village (69.5)...,,Everton North (64.0); Speke (61.9); Clubmoor E...,Stoneycroft (57.6); Springwood (57.2); Orrell ...,Priority exploration council
23,E08000004,Oldham,North West,20,20,57.572562,58.952798,75.034628,5,0,0,0,5,54.517601,53.859367,49.334368,17.853881,0,Failsworth East (75.0); St James' (71.8); Holl...,,,,Priority exploration council
5,E06000050,Cheshire West and Chester,North West,45,45,51.476230,50.706745,71.891740,4,0,7,7,4,54.549985,50.647215,30.758283,37.529767,0,Winsford Wharton (71.9); Winsford Over & Verdi...,,Wolverham (54.0); Westminster (53.6); Blacon (...,Lache (52.9),Priority exploration council
16,E07000125,Rossendale,North West,10,10,63.312220,64.146366,70.247844,4,0,0,1,5,60.859318,69.682980,46.826257,21.563401,3,Haslingden (70.2); Hareholme & Waterfoot (69.2...,,,Whitewell & Stacksteads (58.0),Priority exploration council
12,E07000121,Lancaster,North West,27,27,57.441771,58.236910,78.611324,3,3,1,3,8,48.768641,59.490371,56.102541,4.233553,18,Heysham North (78.6); Poulton (71.8); Bare (69.0),West End (72.6); Scale Hall (72.5); Carnforth ...,Skerton (55.3),Westgate (64.5),Priority exploration council
3,E06000009,Blackpool,North West,21,21,59.994862,59.882908,75.953840,3,0,9,9,5,76.622076,58.965534,23.949898,58.651372,0,Waterloo (76.0); Bloomfield (73.8); Claremont ...,,Victoria (65.5); Talbot (61.8); Hawes Side (58...,Tyldesley (66.4); Layton (59.9),Priority exploration council
30,E08000011,Knowsley,North West,15,15,57.109560,58.340637,69.552800,3,0,9,8,4,77.332241,38.752516,33.670085,55.097641,0,St Gabriels (69.6); Halewood South (68.7); Whi...,,Northwood (59.4); Page Moss (58.3); Stockbridg...,Whiston and Cronton (65.6); Prescot North (63.0),Priority exploration council
24,E08000005,Rochdale,North West,20,20,54.546365,54.778671,70.801412,3,0,3,2,3,58.912574,46.848773,39.489327,42.690006,0,Castleton (70.8); West Middleton (68.4); North...,,North Heywood (60.6); West Heywood (59.0),,Priority exploration council
4,E06000049,Cheshire East,North West,52,52,50.966422,49.924422,68.489939,3,0,3,2,4,48.112340,52.718639,35.246046,34.486854,0,Congleton East (68.5); Crewe St Barnabas (67.6...,,Crewe West (51.1); Macclesfield Hurdsfield (48.4),Crewe North (64.4),Priority exploration council
20,E08000001,Bolton,North West,20,20,58.174833,60.723902,74.769921,3,0,0,0,4,53.666347,59.284254,46.425343,22.707896,0,Farnworth South (74.8); Westhoughton South (66...,,,,Priority exploration council


## 20.9 Map-ready lane file

This file is deliberately slim. Join it to WD25 boundaries using `WD25CD`.

In [10]:
map_cols = [
    "LAD25CD", "LAD25NM", "WD25CD", "WD25NM", "analysis_region",
    "strategic_lane", "strategic_lane_priority",
    "is_clean_watchlist", "is_caveated_watchlist", "is_breakthrough_complacency", "is_demographic_build",
    "initial_watchlist_score", "review_band_clean", "has_major_caveat",
    "dominant_cluster_name", "latest_election_top_party_bucket",
    "demographic_relevance_score", "electoral_opportunity_score", "political_openness_score", "breakthrough_complacency_score",
]

map_cols = [c for c in map_cols if c in review.columns]
map_ready = review[map_cols].copy()

lane_code = {
    "Clean Opportunity": 1,
    "Caveated Opportunity": 2,
    "Breakthrough Build": 3,
    "Long-Term Demographic Build": 4,
    "General Watchlist": 5,
    "Monitor": 6,
}
map_ready["strategic_lane_code"] = map_ready["strategic_lane"].map(lane_code).fillna(99).astype(int)

map_ready.to_csv(OUTPUT_DIR / "north_west_target_review_map_ready_v1.csv", index=False)

print("Map-ready rows:", len(map_ready))
display(map_ready.head())

Map-ready rows: 825


,LAD25CD,LAD25NM,WD25CD,WD25NM,analysis_region,strategic_lane,strategic_lane_priority,is_clean_watchlist,is_caveated_watchlist,is_breakthrough_complacency,is_demographic_build,initial_watchlist_score,review_band_clean,has_major_caveat,dominant_cluster_name,latest_election_top_party_bucket,demographic_relevance_score,electoral_opportunity_score,political_openness_score,breakthrough_complacency_score,strategic_lane_code
103,E06000006,Halton,E05013169,Appleton,North West,Breakthrough Build,3,False,False,True,True,47.412511,Review D Clean,False,Settled Working Families / Skilled Trades Suburbs,lab,82.118212,20.841198,9.675108,77.629534,3
104,E06000006,Halton,E05013170,Bankfield,North West,Breakthrough Build,3,False,False,True,True,47.009626,Review D Clean,False,Post-Industrial Estates / Deprived Working Com...,lab,79.637904,22.182072,9.926951,75.843526,3
105,E06000006,Halton,E05013171,Beechwood & Heath,North West,Monitor,6,False,False,False,False,59.750344,Review D Clean,False,Rooted Older Homeowners,ld,51.296983,67.732524,45.906573,0.000000,6
106,E06000006,Halton,E05013172,Birchfield,North West,Monitor,6,False,False,False,False,36.773439,Review D Clean,False,Stable Suburban Professionals,lab,35.960269,33.589342,16.442167,48.699385,6
107,E06000006,Halton,E05013173,Bridgewater,North West,Breakthrough Build,3,False,False,True,True,50.611330,Review D Clean,False,Post-Industrial Estates / Deprived Working Com...,lab,76.672959,27.068703,22.620732,69.908258,3


## 20.10 Optional all-available outputs

If Notebook 18b generated all-available lane files, this section creates equivalent consolidated national/all-region review files. If the lane files are absent, this section will fall back to the all-available base file and the top 100 only.

In [11]:
if GENERATE_ALL_AVAILABLE_OUTPUTS:
    all_clean_keys = keys_from(all_clean) if all_clean is not None else set()
    all_caveated_keys = keys_from(all_caveated) if all_caveated is not None else set()
    all_demo_keys = keys_from(all_demo) if all_demo is not None else set()
    all_break_keys = keys_from(all_breakthrough) if all_breakthrough is not None else set()
    all_top_keys = keys_from(all_top100) if all_top100 is not None else set()
    
    all_review = add_review_flags(
        base,
        clean_keys=all_clean_keys,
        caveated_keys=all_caveated_keys,
        demo_keys=all_demo_keys,
        break_keys=all_break_keys,
        top100_keys=all_top_keys,
    )
    
    # If all-specific lane files are missing, retain the North West flags only and top100 national flag.
    for col, default in manual_fields.items():
        if col not in all_review.columns:
            all_review[col] = default
    
    all_review["model_review_summary"] = all_review.apply(model_summary, axis=1)
    
    all_export_cols = [c for c in export_cols if c in all_review.columns]
    all_review_export = all_review.sort_values(
        ["strategic_lane_priority", "initial_watchlist_score"],
        ascending=[True, False]
    )[all_export_cols].copy()
    
    all_review_export.to_csv(OUTPUT_DIR / "all_available_consolidated_target_review_v1.csv", index=False)
    
    region_summary = (
        all_review
        .groupby("analysis_region", dropna=False, as_index=False)
        .agg(
            ward_count=("WD25CD", "nunique"),
            mean_score=("initial_watchlist_score", "mean"),
            max_score=("initial_watchlist_score", "max"),
            clean_opportunity_count=("is_clean_watchlist", "sum"),
            caveated_opportunity_count=("is_caveated_watchlist", "sum"),
            breakthrough_build_count=("is_breakthrough_complacency", "sum"),
            demographic_build_count=("is_demographic_build", "sum"),
            top100_count=("is_top100_watchlist", "sum"),
        )
        .sort_values(["top100_count", "mean_score"], ascending=[False, False])
    )
    region_summary.to_csv(OUTPUT_DIR / "all_available_region_review_summary_v1.csv", index=False)
    
    print("All-available consolidated rows:", len(all_review_export))
    display(region_summary)
else:
    print("All-available outputs disabled.")

All-available consolidated rows: 7572


,analysis_region,ward_count,mean_score,max_score,clean_opportunity_count,caveated_opportunity_count,breakthrough_build_count,demographic_build_count,top100_count
6,South West,821,59.059968,81.990184,204,0,1,2,29
0,East Midlands,761,59.985072,81.407491,178,0,6,8,21
7,Wales,762,58.106336,81.143971,146,19,25,34,14
5,South East,1269,53.929143,80.997105,135,0,0,5,11
3,North East,334,59.521788,81.099785,85,0,12,19,9
8,West Midlands,754,55.823551,80.275647,140,0,13,14,6
9,Yorkshire and The Humber,410,53.642055,78.633903,42,0,20,12,4
1,East of England,947,55.177621,76.733766,113,0,9,6,3
4,North West,825,53.179077,79.763938,49,24,64,50,3
2,London,689,32.236373,67.038803,1,0,0,0,0


## 20.11 Manifest

The manifest records what this notebook produced and how each file should be used.

In [ ]:
manifest_rows = [
    ("north_west_consolidated_target_review_v1.csv", "Ward", "Primary consolidated human-review table with all flags and manual fields."),
    ("north_west_clean_opportunity_wards_v1.csv", "Ward", "Clean Review A/B opportunity wards with no major caveat."),
    ("north_west_caveated_opportunity_wards_v1.csv", "Ward", "High-scoring wards with caveats, especially CED-derived or boundary issues."),
    ("north_west_breakthrough_build_wards_v1.csv", "Ward", "Safe-seat / low-turnout / demographically plausible breakthrough wards."),
    ("north_west_long_term_demographic_build_wards_v1.csv", "Ward", "High demographic relevance but weaker immediate electoral opportunity."),
    ("north_west_council_briefing_summary_v1.csv", "Council", "Council-level briefing summary with top wards and strategic interpretation."),
    ("north_west_target_review_map_ready_v1.csv", "Ward map", "Slim file for mapping strategic lanes by WD25CD."),
    ("all_available_consolidated_target_review_v1.csv", "Ward", "Optional all-region consolidated review file, if enabled."),
    ("all_available_region_review_summary_v1.csv", "Region", "Optional regional summary, if enabled."),
]

manifest = pd.DataFrame(manifest_rows, columns=["file", "level", "purpose"])
manifest["created_by"] = "20_consolidated_target_review_pack_v1.ipynb"
manifest["created_date"] = str(date.today())
manifest.to_csv(OUTPUT_DIR / "target_review_pack_manifest_v1.csv", index=False)

display(manifest)
print("Output folder:", OUTPUT_DIR)

,file,level,purpose,created_by,created_date
0,north_west_consolidated_target_review_v1.csv,Ward,Primary consolidated human-review table with a...,20_consolidated_target_review_pack_v1.ipynb,2026-05-27
1,north_west_clean_opportunity_wards_v1.csv,Ward,Clean Review A/B opportunity wards with no maj...,20_consolidated_target_review_pack_v1.ipynb,2026-05-27
2,north_west_caveated_opportunity_wards_v1.csv,Ward,"High-scoring wards with caveats, especially CE...",20_consolidated_target_review_pack_v1.ipynb,2026-05-27
3,north_west_breakthrough_build_wards_v1.csv,Ward,Safe-seat / low-turnout / demographically plau...,20_consolidated_target_review_pack_v1.ipynb,2026-05-27
4,north_west_long_term_demographic_build_wards_v...,Ward,High demographic relevance but weaker immediat...,20_consolidated_target_review_pack_v1.ipynb,2026-05-27
5,north_west_council_briefing_summary_v1.csv,Council,Council-level briefing summary with top wards ...,20_consolidated_target_review_pack_v1.ipynb,2026-05-27
6,north_west_target_review_map_ready_v1.csv,Ward map,Slim file for mapping strategic lanes by WD25CD.,20_consolidated_target_review_pack_v1.ipynb,2026-05-27
7,all_available_consolidated_target_review_v1.csv,Ward,"Optional all-region consolidated review file, ...",20_consolidated_target_review_pack_v1.ipynb,2026-05-27
8,all_available_region_review_summary_v1.csv,Region,"Optional regional summary, if enabled.",20_consolidated_target_review_pack_v1.ipynb,2026-05-27


Output folder: c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_review_pack_v1


: 

## 20.12 Next step

Use the consolidated review pack for manual political validation.

Before calling any ward a target, add local knowledge:

- known candidate availability;
- local SDP/member presence;
- recent activity;
- issue hook;
- delivery/canvassing practicality;
- manual priority notes.

The next notebook/reporting step should use the consolidated review pack and map-ready file to produce the first professional North West Target Intelligence briefing.